# ADK — Complex Agent

**Framework:** Google Agent Development Kit (ADK)  
**Level:** 03 — Complex  
**Model:** `gemini-2.0-flash`

### What's New vs Intermediate
| Feature | Intermediate | Complex |
|---|---|---|
| Behavior | Reactive (answers questions) | **Proactive** (plans → acts → self-critiques) |
| Planning | None | **Explicit step 1: write a plan** |
| Quality control | None | **Reflexion: score_report() → rewrite if < 7** |
| Output visibility | Final response only | **Streaming all intermediate events** |
| Goal | Single city briefing | **Multi-city comparison + ranking** |

## Concept: Plan → Gather → Critique → Refine

```
User: "Compare Tokyo, Paris, Bangalore"
              │
              ▼
┌─────────────────────────────────────────────────────────┐
│  STEP 1: PLAN (LLM output, no tool)                     │
│  "I will research Tokyo, Paris, Bangalore..."           │
└────────────────────┬────────────────────────────────────┘
                     │
┌────────────────────▼────────────────────────────────────┐
│  STEP 2: GATHER (3 tool calls × 3 cities = 9 calls)     │
│  get_weather(Tokyo) + get_time(Tokyo) + advisory(Tokyo) │
│  get_weather(Paris) + ...                               │
│  get_weather(Bangalore) + ...                           │
└────────────────────┬────────────────────────────────────┘
                     │
┌────────────────────▼────────────────────────────────────┐
│  STEP 3: DRAFT REPORT (LLM synthesizes all data)        │
└────────────────────┬────────────────────────────────────┘
                     │
┌────────────────────▼────────────────────────────────────┐
│  STEP 4: CRITIQUE — score_report(draft)                 │
│  score < 7 → revise and resubmit                        │
│  score >= 7 → deliver final report                      │
└─────────────────────────────────────────────────────────┘
```

**Key insight:** The agent uses a **`score_report` tool** to self-assess. This is a form of **Reflexion** — instead of a separate critic LLM, the same agent evaluates its own work. The instruction forces a feedback loop without requiring any extra code outside the agent definition.

## Setup

In [ ]:
import asyncio
import os
from dotenv import load_dotenv

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## Tools — Including `score_report` for Self-Assessment

In [ ]:
def get_weather(city: str) -> dict:
    """Get weather and a 1-10 weather score for a city.
    Args:
        city: City name.
    Returns:
        Weather dict with condition, temperature, and score.
    """
    data = {
        "london":    {"condition": "Cloudy",       "temp_c": 12, "score": 5},
        "new york":  {"condition": "Sunny",         "temp_c": 22, "score": 8},
        "bangalore": {"condition": "Rainy",         "temp_c": 25, "score": 6},
        "tokyo":     {"condition": "Clear",         "temp_c": 18, "score": 9},
        "paris":     {"condition": "Partly Cloudy", "temp_c": 16, "score": 7},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No data for '{city}'."}


def get_time(city: str) -> dict:
    """Get local time for a city.
    Args:
        city: City name.
    Returns:
        Dict with local_time.
    """
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    return {"city": city, "local_time": times.get(city.lower(), "Unknown")}


def get_travel_advisory(city: str) -> dict:
    """Get travel safety advisory and safety score for a city.
    Args:
        city: City name.
    Returns:
        Advisory dict with level, notes, and safety_score.
    """
    data = {
        "london":    {"level": "Low",    "notes": "Routine precautions.",      "safety_score": 8},
        "new york":  {"level": "Low",    "notes": "Normal precautions.",       "safety_score": 7},
        "bangalore": {"level": "Medium", "notes": "Monsoon affects transport.", "safety_score": 6},
        "tokyo":     {"level": "Low",    "notes": "Very safe city.",            "safety_score": 10},
        "paris":     {"level": "Low",    "notes": "Alert in crowded spots.",    "safety_score": 8},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No advisory for '{city}'."}


def score_report(report: str) -> dict:
    """Evaluate the quality of a travel comparison report.

    Call this BEFORE delivering your final response.
    If score < 7, rewrite addressing the suggestions, then call this again.
    If score >= 7, deliver the report as your final answer.

    Args:
        report: The full text of your draft report.
    Returns:
        Dict with score, pass/fail, and suggestions.
    """
    score = 0
    suggestions = []

    if any(kw in report.lower() for kw in ["rank", "1.", "recommended", "best"]):
        score += 3
    else:
        suggestions.append("Add a clear ranking or top recommendation.")

    if any(kw in report.lower() for kw in ["°c", "celsius", "temperature", "weather"]):
        score += 2
    else:
        suggestions.append("Include specific temperature/weather data.")

    if any(kw in report.lower() for kw in ["advisory", "safety", "safe"]):
        score += 2
    else:
        suggestions.append("Include safety advisory information.")

    if len(report) > 300:
        score += 2
    else:
        suggestions.append("Report is too brief — add more detail.")

    if any(kw in report.lower() for kw in ["gmt", "ist", "jst", "est", "cet", "time"]):
        score += 1

    return {
        "score": min(score, 10),
        "threshold": 7,
        "pass": score >= 7,
        "suggestions": suggestions if suggestions else ["Report looks great!"],
    }


print("4 tools defined: get_weather, get_time, get_travel_advisory, score_report")
# Demo the scorer
print("\nScoring a bad draft:", score_report("Tokyo is nice."))
print("Scoring a good draft:", score_report(
    "1. Tokyo: Clear, 18°C. Safety: Low advisory. Time: 22:30 JST. "
    "2. Paris: Partly Cloudy, 16°C. Recommended for best combination of weather and safety. " * 5
))

## Complex Agent Definition

The instruction is the key — it enforces a **4-step procedure** with a built-in self-critique loop.

In [ ]:
agent = Agent(
    name="trip_planner",
    model="gemini-2.0-flash",
    description="A trip planner that produces ranked city comparison reports.",
    instruction="""You are an expert travel analyst. When given cities to compare, follow these steps:

STEP 1 — PLAN: Before calling any tools, write:
  'PLAN: I will research [cities], checking weather, time, and safety for each.'

STEP 2 — GATHER: Call get_weather, get_time, and get_travel_advisory for EVERY city.
  Do not skip any city or any tool.

STEP 3 — DRAFT: Write a comparison report ranking the cities.
  Include: weather scores, safety scores, best time to visit, final ranking.

STEP 4 — CRITIQUE: Call score_report() with your draft.
  If score < 7: revise the report addressing the suggestions, then check the score again.
  If score >= 7: deliver the final report as your response.

You MUST complete all 4 steps. Never skip the critique step.""",
    tools=[get_weather, get_time, get_travel_advisory, score_report],
)

print(f"Complex agent '{agent.name}' — {len(agent.tools)} tools")

## Run with Streaming

At the Complex level, we stream all intermediate events — not just the final response. This lets you watch every tool call as it happens.

In [ ]:
async def run_with_streaming(query: str) -> str:
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="trip_planner", user_id="user_01"
    )
    runner = InMemoryRunner(agent=agent, session_service=session_service)

    print(f"Query: {query}\n")
    print("--- Streaming trace ---")

    final_response = ""
    async for event in runner.run_async(
        user_id=session.user_id,
        session_id=session.id,
        new_message=genai_types.Content(
            role="user", parts=[genai_types.Part(text=query)]
        ),
    ):
        # Capture different event types
        if hasattr(event, "tool_call") and event.tool_call:
            tc = event.tool_call
            args_str = str(tc.args)[:60] if tc.args else ""
            print(f"  ▶ tool_call   : {tc.name}({args_str})")
        elif hasattr(event, "tool_result") and event.tool_result:
            result_str = str(event.tool_result)[:80]
            print(f"  ◀ tool_result : {result_str}")
        elif event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    final_response += part.text

    print("\n--- Final Report ---")
    return final_response


query = "Compare Tokyo, Paris, and Bangalore for travel. Rank them and give a final recommendation."
report = await run_with_streaming(query)
print(report)

In [ ]:
# Count tool calls in the response
print(f"Report length: {len(report)} characters")
print(f"Contains ranking: {'1.' in report or 'rank' in report.lower()}")
print(f"Contains temperatures: {'°C' in report or 'celsius' in report.lower()}")
print(f"Contains safety info: {'advisory' in report.lower() or 'safe' in report.lower()}")

# Manually score the output to verify the reflexion worked
final_score = score_report(report)
print(f"\nFinal quality score: {final_score['score']}/10")
print(f"Pass threshold met: {final_score['pass']}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| 4-step instruction | Instruction engineering for complex behaviors — no framework code needed |
| `score_report` as a tool | Reflexion via tool: agent calls it, reads score, decides whether to revise |
| Streaming events | Observe every tool call in real-time — essential for debugging long tasks |
| Multi-city goal decomposition | Agent automatically loops over all cities without explicit loop code |

### ADK Reflexion Pattern vs LangGraph
| Approach | ADK (this notebook) | LangGraph (03-Complex) |
|---|---|---|
| Critique mechanism | `score_report` tool in the agent | Dedicated `critic_node` in the graph |
| Retry logic | Encoded in the instruction | Explicit conditional edge |
| Visibility | Streaming events | `graph.stream()` node transitions |
| Control | LLM decides when to stop | Graph enforces max retries |

**ADK's approach is simpler but less controllable.** LangGraph's graph gives you hard guarantees on retry limits and flow.

### Next Steps
- Try `vertex-ai-real-world/` — same agent on Vertex AI with real Gemini on GCP
- Compare this notebook's reflexion pattern with [LangGraph 03-Complex](../../LangGraph/03-complex/agent.ipynb)